## Goal - Create reproducible training and validation datasets.

Output:
    X_train,
    X_val,
    y_train,
    y_val

No preprocessing is performed in this notebook.

In [1]:
# Required Imports
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [2]:
# Loading the data
train_df = pd.read_csv("../data/raw/train.csv")
test_df = pd.read_csv("../data/raw/test.csv")

In [3]:
# Splitting into features X and label y
X = train_df.drop("Survived", axis=1)
y = train_df["Survived"]

In [4]:
# Train/Validation Split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [5]:
print(X_train.shape)
print(X_val.shape)
print(y_train.shape)
print(y_val.shape)

(712, 11)
(179, 11)
(712,)
(179,)


In [6]:
## Checking Class Imbalance
y_train.value_counts(normalize=True)

Survived
0    0.616573
1    0.383427
Name: proportion, dtype: float64

In [7]:
y_val.value_counts(normalize=True)

Survived
0    0.614525
1    0.385475
Name: proportion, dtype: float64

# Production Feature Engineering

In this section, we validate the production implementation of our custom feature engineering transformers.

In [8]:
from titanic_survival.features.engineering import TitleExtractor

In [9]:
title_extractor = TitleExtractor()

In [10]:
X_train_title = title_extractor.fit_transform(X_train)
X_val_title = title_extractor.fit_transform(X_val)

In [11]:
X_train_title[["Name", "Title"]].head(10)

,Name,Title
692,"Lam, Mr. Ali",Mr
481,"Frost, Mr. Anthony Wood ""Archie""",Mr
527,"Farthing, Mr. John",Mr
855,"Aks, Mrs. Sam (Leah Rosen)",Mrs
801,"Collyer, Mrs. Harvey (Charlotte Annie Tate)",Mrs
652,"Kalvik, Mr. Johannes Halvorsen",Mr
509,"Lang, Mr. Fang",Mr
557,"Robbins, Mr. Victor",Mr
828,"McCormack, Mr. Thomas Joseph",Mr
18,"Vander Planke, Mrs. Julius (Emelia Maria Vande...",Mrs


In [12]:
X_train_title["Title"].value_counts()

Title
Mr          412
Miss        141
Mrs         107
Master       31
Dr            6
Rev           5
Col           2
Mlle          2
Major         1
Lady          1
Sir           1
Ms            1
Jonkheer      1
Don           1
Name: count, dtype: int64

In [13]:
X_val_title[["Name", "Title"]].head(10)

,Name,Title
565,"Davies, Mr. Alfred J",Mr
160,"Cribb, Mr. John Hatfield",Mr
553,"Leeni, Mr. Fahim (""Philip Zenni"")",Mr
860,"Hansen, Mr. Claus Peter",Mr
241,"Murphy, Miss. Katherine ""Kate""",Miss
559,"de Messemaeker, Mrs. Guillaume Joseph (Emma)",Mrs
387,"Buss, Miss. Kate",Miss
536,"Butt, Major. Archibald Willingham",Major
698,"Thayer, Mr. John Borland",Mr
99,"Kantor, Mr. Sinai",Mr


In [14]:
X_val_title["Title"].value_counts()

Title
Mr              105
Miss             41
Mrs              18
Master            9
Major             1
Mme               1
Capt              1
the Countess      1
Dr                1
Rev               1
Name: count, dtype: int64

In [15]:
from titanic_survival.features.engineering import FamilySizeCreator
from titanic_survival.features.engineering import IsAloneCreator
from titanic_survival.features.engineering import DeckExtractor
from titanic_survival.features.engineering import TicketPrefixExtractor

In [16]:
family_size_creator = FamilySizeCreator()
is_alone_creator = IsAloneCreator()
deck_extractor = DeckExtractor()
ticket_prefix_extractor = TicketPrefixExtractor()

In [17]:
X_train_family_size = family_size_creator.fit_transform(X_train)
X_val_family_size = family_size_creator.fit_transform(X_val)

X_train_is_alone = is_alone_creator.fit_transform(X_train_family_size)
X_val_is_alone = is_alone_creator.fit_transform(X_val_family_size)

X_train_deck = deck_extractor.fit_transform(X_train)
X_val_deck = deck_extractor.fit_transform(X_val)

X_train_ticket_prefix = ticket_prefix_extractor.fit_transform(X_train)
X_val_ticket_prefix = ticket_prefix_extractor.fit_transform(X_val)

In [18]:
X_train_family_size["FamilySize"].value_counts(dropna=False, normalize=True)

FamilySize
1     0.609551
2     0.181180
3     0.108146
4     0.030899
6     0.022472
5     0.019663
7     0.015449
11    0.007022
8     0.005618
Name: proportion, dtype: float64

In [19]:
X_val_family_size["FamilySize"].value_counts(dropna=False, normalize=True)

FamilySize
1     0.575419
2     0.178771
3     0.139665
4     0.039106
6     0.033520
8     0.011173
11    0.011173
5     0.005587
7     0.005587
Name: proportion, dtype: float64

In [20]:
X_train_is_alone["IsAlone"].value_counts(dropna=False, normalize=True)

IsAlone
1    0.609551
0    0.390449
Name: proportion, dtype: float64

In [21]:
X_val_is_alone["IsAlone"].value_counts(dropna=False, normalize=True)

IsAlone
1    0.575419
0    0.424581
Name: proportion, dtype: float64

In [22]:
X_train_deck["Deck"].value_counts(dropna=False, normalize=True)

Deck
Unknown    0.775281
C          0.057584
B          0.047753
E          0.040730
D          0.036517
A          0.019663
F          0.015449
G          0.005618
T          0.001404
Name: proportion, dtype: float64

In [23]:
X_val_deck["Deck"].value_counts(dropna=False, normalize=True)

Deck
Unknown    0.754190
C          0.100559
B          0.072626
D          0.039106
E          0.016760
F          0.011173
A          0.005587
Name: proportion, dtype: float64

In [24]:
X_train_ticket_prefix["TicketPrefix"].value_counts(dropna=False, normalize=True)

TicketPrefix
NONE       0.733146
PC         0.075843
CA         0.044944
A5         0.028090
SOTONOQ    0.015449
STONO      0.014045
WC         0.014045
SCPARIS    0.012640
A4         0.008427
C          0.007022
SOC        0.007022
STONO2     0.005618
LINE       0.004213
FCC        0.004213
PP         0.004213
PPP        0.002809
SCAH       0.002809
SOTONO2    0.002809
SOPP       0.002809
SWPP       0.001404
SOP        0.001404
FA         0.001404
SCA4       0.001404
WEP        0.001404
AS         0.001404
SP         0.001404
Name: proportion, dtype: float64

In [25]:
X_val_ticket_prefix["TicketPrefix"].value_counts(dropna=False, normalize=True)

TicketPrefix
NONE       0.776536
CA         0.050279
PC         0.033520
SOTONOQ    0.022346
STONO2     0.011173
SCPARIS    0.011173
STONO      0.011173
WEP        0.011173
FCC        0.011173
A4         0.005587
SOC        0.005587
SCOW       0.005587
A5         0.005587
SWPP       0.005587
LINE       0.005587
SOPP       0.005587
FC         0.005587
SC         0.005587
CASOTON    0.005587
SCAH       0.005587
Name: proportion, dtype: float64

In [26]:
from titanic_survival.features.engineering import TicketGroupSizeCreator

In [27]:
tgs_transformer = TicketGroupSizeCreator()
X_train_tgs = tgs_transformer.fit_transform(X_train)
X_val_tgs = tgs_transformer.transform(X_val)

In [29]:
X_train_tgs[["Ticket", "TicketGroupSize"]]

,Ticket,TicketGroupSize
692,1601,6
481,239854,1
527,PC 17483,1
855,392091,1
801,C.A. 31921,2
...,...,...
359,330980,1
258,PC 17755,2
736,W./C. 6608,4
462,111320,1


In [31]:
X_val_tgs[["Ticket", "TicketGroupSize"]]

,Ticket,TicketGroupSize
565,A/4 48871,1
160,371362,1
553,2620,1
860,350026,1
241,367230,1
...,...,...
880,230433,1
91,347466,1
883,C.A./SOTON 34068,1
473,SC/AH Basle 541,1


In [35]:
from titanic_survival.pipelines.engineering import feature_engineering_pipeline

In [36]:
X_train_transformed = feature_engineering_pipeline.fit_transform(X_train)

In [37]:
X_train_transformed.head(10)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title,FamilySize,IsAlone,Deck,TicketPrefix,TicketGroupSize
692,693,3,"Lam, Mr. Ali",male,NaN,0,0,1601,56.4958,NaN,S,Mr,1,1,Unknown,NONE,6
481,482,2,"Frost, Mr. Anthony Wood ""Archie""",male,NaN,0,0,239854,0.0000,NaN,S,Mr,1,1,Unknown,NONE,1
527,528,1,"Farthing, Mr. John",male,NaN,0,0,PC 17483,221.7792,C95,S,Mr,1,1,C,PC,1
855,856,3,"Aks, Mrs. Sam (Leah Rosen)",female,18.0,0,1,392091,9.3500,NaN,S,Mrs,2,0,Unknown,NONE,1
801,802,2,"Collyer, Mrs. Harvey (Charlotte Annie Tate)",female,31.0,1,1,C.A. 31921,26.2500,NaN,S,Mrs,3,0,Unknown,CA,2
652,653,3,"Kalvik, Mr. Johannes Halvorsen",male,21.0,0,0,8475,8.4333,NaN,S,Mr,1,1,Unknown,NONE,1
509,510,3,"Lang, Mr. Fang",male,26.0,0,0,1601,56.4958,NaN,S,Mr,1,1,Unknown,NONE,6
557,558,1,"Robbins, Mr. Victor",male,NaN,0,0,PC 17757,227.5250,NaN,C,Mr,1,1,Unknown,PC,4
828,829,3,"McCormack, Mr. Thomas Joseph",male,NaN,0,0,367228,7.7500,NaN,Q,Mr,1,1,Unknown,NONE,1
18,19,3,"Vander Planke, Mrs. Julius (Emelia Maria Vande...",female,31.0,1,0,345763,18.0000,NaN,S,Mrs,2,0,Unknown,NONE,1
